# SmartCampus AI — Multimodal Evaluation: VLM OCR (`1.4 Reader`) & Whisper STT (`1.1 Listener`)

**Project:** SmartCampus AI (IIT / ISB + ISA extension, Groupe 3 *Night Agents*, NAU 2026–2027)
**Recommended Colab Pro Runtime:** GPU `L4 (24 GB)` (Account #5)

This notebook evaluates our open-source <= 8B multimodal stack:
1. **VLM OCR (`VISION 1.4 Reader`):** Zero-Shot & Few-Shot extraction of bilingual French/Arabic Tunisian Baccalaureate & Licence transcripts comparing **`Qwen/Qwen2.5-VL-7B-Instruct`** (7.6B, Apache 2.0) vs **`google/gemma-3-4b-it`** (4.3B) against `SC V3 §10` target (`>= 95%` field-level exact match).
2. **Whisper STT (`VISION 1.1 Listener`):** Word Error Rate (`WER`) baseline on Tunisian Derja / French / code-switched student voice notes using **`openai/whisper-large-v3`**.


In [ ]:
# Cell 1 — Mount Google Drive & install dependencies
from google.colab import drive
drive.mount("/content/drive")

!pip install -q bitsandbytes accelerate transformers qwen-vl-utils pillow pypdf jiwer openai-whisper pandas


In [ ]:
# Cell 2 — Configuration & Paths
import json, time
from pathlib import Path
import pandas as pd
import torch
from PIL import Image

DRIVE_ROOT = Path("/content/drive/MyDrive/SmartCampus_AI")
TRANSCRIPTS_DIR = DRIVE_ROOT / "eval_data" / "transcripts"   # Put anonymized PNG/JPG + ground_truth.json here
AUDIO_DIR = DRIVE_ROOT / "eval_data" / "voice_notes"         # Put anonymized WAV/MP3 + ground_truth.json here
OUT_DIR = DRIVE_ROOT / "benchmarks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VLM_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # Compare against: "google/gemma-3-4b-it"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Cell 3 — Load Open-Source VLM (4-bit NF4) & Define Strict JSON OCR Prompt
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
)

OCR_SYSTEM_PROMPT = (
    "You are VISION 1.4 Reader, a high-precision OCR extractor for Tunisian Baccalaureate "
    "and Licence grade transcripts (bilingual French & Arabic). Return ONLY valid JSON with keys: "
    "bac_section, session_year, general_average, mention, subjects (list of {name, grade, coefficient, confidence}). "
    "Never invent unreadable grades; set low confidence (< 0.85) for blurry fields so a human reviews them."
)

def extract_transcript_json(image_path: Path) -> tuple[str, float]:
    img = Image.open(image_path).convert("RGB")
    img.thumbnail((1280, 1280))
    messages = [
        {"role": "system", "content": OCR_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": "Extract all grades, coefficients, section, and general average as strict JSON."},
            ],
        },
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], padding=True, return_tensors="pt").to(vlm_model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=1024, temperature=0.0, do_sample=False)
    latency_s = round(time.perf_counter() - t0, 3)
    trimmed = generated_ids[:, inputs.input_ids.shape[1]:]
    output_text = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    return output_text, latency_s


In [ ]:
# Cell 4 — Run VLM OCR Field-Level Evaluation (when anonymized transcript fixtures exist)
gt_file = TRANSCRIPTS_DIR / "ground_truth.json"
if gt_file.exists():
    gt_items = json.loads(gt_file.read_text(encoding="utf-8"))
    results = []
    for item in gt_items:
        img_path = TRANSCRIPTS_DIR / item["file"]
        raw_out, lat = extract_transcript_json(img_path)
        try:
            parsed = json.loads(raw_out.strip("`").replace("json\n", "", 1))
            avg_match = abs(float(parsed.get("general_average", -1)) - float(item["general_average"])) <= 0.05
            sec_match = str(parsed.get("bac_section", "")).lower() == str(item["bac_section"]).lower()
        except Exception:
            avg_match, sec_match = False, False
        results.append({
            "model": VLM_MODEL_ID,
            "file": item["file"],
            "latency_s": lat,
            "general_average_exact": avg_match,
            "bac_section_exact": sec_match,
        })
    df_vlm = pd.DataFrame(results)
    out_csv = OUT_DIR / f"vlm_ocr_{VLM_MODEL_ID.replace('/', '__')}.csv"
    df_vlm.to_csv(out_csv, index=False)
    print("Saved VLM OCR benchmark:", out_csv)
    display(df_vlm)
else:
    print("Place anonymized transcripts + ground_truth.json in:", TRANSCRIPTS_DIR)


In [ ]:
# Cell 5 — Whisper STT (whisper-large-v3) WER Benchmark on Tunisian Voice Notes
import whisper
from jiwer import wer

whisper_gt = AUDIO_DIR / "ground_truth.json"
if whisper_gt.exists():
    stt_model = whisper.load_model("large-v3")
    items = json.loads(whisper_gt.read_text(encoding="utf-8"))
    stt_rows = []
    for item in items:
        audio_path = AUDIO_DIR / item["file"]
        t0 = time.perf_counter()
        res = stt_model.transcribe(
            str(audio_path),
            initial_prompt="SmartCampus NAU Sfax IIT ISB ISA Licence Prépa Ingénieur GLID SDIA عسلامة شنوة كيفاش",
        )
        lat = round(time.perf_counter() - t0, 3)
        hyp = res["text"].strip()
        ref = item["reference_text"].strip()
        stt_rows.append({
            "file": item["file"],
            "latency_s": lat,
            "wer": round(wer(ref, hyp), 4),
            "reference": ref,
            "hypothesis": hyp,
        })
    df_stt = pd.DataFrame(stt_rows)
    out_stt = OUT_DIR / "whisper_large_v3_wer.csv"
    df_stt.to_csv(out_stt, index=False)
    print(f"Mean WER: {df_stt['wer'].mean():.4f} | Saved:", out_stt)
    display(df_stt[["file", "latency_s", "wer"]])
else:
    print("Place anonymized audio notes + ground_truth.json in:", AUDIO_DIR)
